# 03 — Experimento Comparativo
Para cada técnica se prueban 3 hiperparámetros con 2 valores cada uno (8 combinaciones por técnica = 24 runs totales).

In [1]:
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from itertools import product

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit

from preprocessing import encode_and_split
from paths import ensure_results_dirs
from plots import (
    plot_experiment_bars, plot_confusion_matrix,
    plot_confusion_matrices_combined, plot_metrics_bar
)
from evaluation import evaluate
from experiment_analysis import generate_comparative_analysis
from best_models import (
    select_best_configs, save_best_configs, train_best_models, SVM_SAMPLE
)

PROCESSED_PATH = '../data/processed/datos_limpios.csv'
RESULTS_PATH   = '../results/'
CM_PATH        = '../results/confusion_matrices/'

ensure_results_dirs('../results')
print('Librerías cargadas')

Librerías cargadas


## 1. Carga de datos

In [2]:
# Reutilizamos el mismo split del notebook 02 (mismo random_state=42)
# para que los resultados del experimento sean comparables entre técnicas.
df = pd.read_csv(PROCESSED_PATH)
X_train, X_test, X_train_sc, X_test_sc, y_train, y_test, le = encode_and_split(df)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Clases: {le.classes_}')

Train: 171,341 | Test: 42,836
Clases: ['Ideación suicida' 'Intento de Suicidio']


## 2. Diseño experimental
| Técnica | Hiperparámetro 1 | Hiperparámetro 2 | Hiperparámetro 3 |
|---------|-----------------|-----------------|------------------|
| RF | `n_estimators` [100, 200] | `max_depth` [10, 20] | `min_samples_split` [5, 10] |
| SVM | `C` [1, 10] | `kernel` [rbf, poly] | `gamma` [scale, auto] |
| MLP | `hidden_layer_sizes` [(100,50), (200,100)] | `activation` [relu, tanh] | `learning_rate_init` [0.001, 0.01] |

> Diseño factorial completo: 2³ = 8 combinaciones por técnica → **24 runs totales**

## 3. Experimento Random Forest

In [3]:
# Experimento RF: itertools.product genera las 8 combinaciones (2³)
# de los 3 hiperparámetros con 2 valores cada uno.
# class_weight='balanced' se mantiene fijo en todas las corridas
# para que el efecto medido sea solo de los hiperparámetros experimentales.
rf_params = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20],
    'min_samples_split': [5, 10]
}

rf_results = []
for n_est, depth, split in product(*rf_params.values()):
    model = RandomForestClassifier(
        n_estimators=n_est, max_depth=depth, min_samples_split=split,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rf_results.append({
        'Técnica': 'RF',
        'n_estimators': n_est,
        'max_depth': depth,
        'min_samples_split': split,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1 (weighted)': round(f1_score(y_test, y_pred, average='weighted'), 4)
    })
    print(f"  RF | n={n_est} | d={depth} | s={split} → Acc={rf_results[-1]['Accuracy']}")

df_rf = pd.DataFrame(rf_results)
print('\n✅ RF completado')
print(df_rf[['n_estimators','max_depth','min_samples_split','Accuracy','F1 (weighted)']].to_string(index=False))

  RF | n=100 | d=10 | s=5 → Acc=0.6775
  RF | n=100 | d=10 | s=10 → Acc=0.6817
  RF | n=100 | d=20 | s=5 → Acc=0.6774
  RF | n=100 | d=20 | s=10 → Acc=0.6778
  RF | n=200 | d=10 | s=5 → Acc=0.6792
  RF | n=200 | d=10 | s=10 → Acc=0.6815
  RF | n=200 | d=20 | s=5 → Acc=0.6788
  RF | n=200 | d=20 | s=10 → Acc=0.6779

✅ RF completado
 n_estimators  max_depth  min_samples_split  Accuracy  F1 (weighted)
          100         10                  5    0.6775         0.6833
          100         10                 10    0.6817         0.6861
          100         20                  5    0.6774         0.6823
          100         20                 10    0.6778         0.6840
          200         10                  5    0.6792         0.6844
          200         10                 10    0.6815         0.6864
          200         20                  5    0.6788         0.6837
          200         20                 10    0.6779         0.6842


## 4. Experimento SVM

In [4]:
# La muestra estratificada se crea UNA SOLA VEZ y se reutiliza en las 8 corridas del SVM.
# Esto garantiza que la diferencia en resultados se deba solo a los hiperparámetros
# y no a variaciones en la muestra de entrenamiento.
from sklearn.model_selection import StratifiedShuffleSplit

SVM_SAMPLE = 15_000

svm_params = {
    'C':      [1, 10],
    'kernel': ['rbf', 'poly'],
    'gamma':  ['scale', 'auto']
}

sss = StratifiedShuffleSplit(n_splits=1, train_size=SVM_SAMPLE, random_state=42)
svm_idx, _ = next(sss.split(X_train_sc, y_train))
X_tr_svm = X_train_sc[svm_idx]
y_tr_svm = y_train.iloc[svm_idx]

svm_results = []
for c, kernel, gamma in product(*svm_params.values()):
    model = SVC(C=c, kernel=kernel, gamma=gamma, class_weight='balanced', random_state=42)
    model.fit(X_tr_svm, y_tr_svm)
    y_pred = model.predict(X_test_sc)
    svm_results.append({
        'Técnica': 'SVM',
        'C': c,
        'kernel': kernel,
        'gamma': gamma,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1 (weighted)': round(f1_score(y_test, y_pred, average='weighted'), 4)
    })
    print(f"  SVM | C={c} | k={kernel} | g={gamma} → Acc={svm_results[-1]['Accuracy']}")

df_svm = pd.DataFrame(svm_results)
print('\n✅ SVM completado')
print(df_svm[['C','kernel','gamma','Accuracy','F1 (weighted)']].to_string(index=False))

  SVM | C=1 | k=rbf | g=scale → Acc=0.6631
  SVM | C=1 | k=rbf | g=auto → Acc=0.6632
  SVM | C=1 | k=poly | g=scale → Acc=0.6611
  SVM | C=1 | k=poly | g=auto → Acc=0.6611
  SVM | C=10 | k=rbf | g=scale → Acc=0.6427
  SVM | C=10 | k=rbf | g=auto → Acc=0.6427
  SVM | C=10 | k=poly | g=scale → Acc=0.6594
  SVM | C=10 | k=poly | g=auto → Acc=0.6594

✅ SVM completado
 C kernel gamma  Accuracy  F1 (weighted)
 1    rbf scale    0.6631         0.6694
 1    rbf  auto    0.6632         0.6694
 1   poly scale    0.6611         0.6670
 1   poly  auto    0.6611         0.6670
10    rbf scale    0.6427         0.6531
10    rbf  auto    0.6427         0.6531
10   poly scale    0.6594         0.6650
10   poly  auto    0.6594         0.6650


## 5. Experimento MLP

In [5]:
# Para MLP usamos 'activation' como tercer hiperparámetro (relu vs tanh)
# en lugar de max_iter, porque activation tiene mayor impacto en el aprendizaje:
#   - relu: no tiene problema de gradiente que desaparece, más rápida de calcular
#   - tanh: salida centrada en 0, puede converger mejor en ciertos problemas
# early_stopping=True se mantiene fijo para que la comparación sea justa.
mlp_params = {
    'hidden_layer_sizes': [(100, 50), (200, 100)],
    'activation':         ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01]
}

mlp_results = []
for layers, act, lr in product(*mlp_params.values()):
    model = MLPClassifier(
        hidden_layer_sizes=layers, activation=act, learning_rate_init=lr,
        max_iter=200, random_state=42,
        early_stopping=True, validation_fraction=0.1, verbose=False
    )
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    mlp_results.append({
        'Técnica': 'MLP',
        'hidden_layers': str(layers),
        'activation': act,
        'learning_rate': lr,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1 (weighted)': round(f1_score(y_test, y_pred, average='weighted'), 4)
    })
    print(f"  MLP | layers={layers} | act={act} | lr={lr} → Acc={mlp_results[-1]['Accuracy']}")

df_mlp = pd.DataFrame(mlp_results)
print('\n✅ MLP completado')
print(df_mlp[['hidden_layers','activation','learning_rate','Accuracy','F1 (weighted)']].to_string(index=False))

  MLP | layers=(100, 50) | act=relu | lr=0.001 → Acc=0.7355
  MLP | layers=(100, 50) | act=relu | lr=0.01 → Acc=0.7345
  MLP | layers=(100, 50) | act=tanh | lr=0.001 → Acc=0.736
  MLP | layers=(100, 50) | act=tanh | lr=0.01 → Acc=0.7347
  MLP | layers=(200, 100) | act=relu | lr=0.001 → Acc=0.7353
  MLP | layers=(200, 100) | act=relu | lr=0.01 → Acc=0.7319
  MLP | layers=(200, 100) | act=tanh | lr=0.001 → Acc=0.7357
  MLP | layers=(200, 100) | act=tanh | lr=0.01 → Acc=0.7297

✅ MLP completado
hidden_layers activation  learning_rate  Accuracy  F1 (weighted)
    (100, 50)       relu          0.001    0.7355         0.6852
    (100, 50)       relu          0.010    0.7345         0.6781
    (100, 50)       tanh          0.001    0.7360         0.6837
    (100, 50)       tanh          0.010    0.7347         0.6867
   (200, 100)       relu          0.001    0.7353         0.6799
   (200, 100)       relu          0.010    0.7319         0.6678
   (200, 100)       tanh          0.001    0.735

## 6. Tabla consolidada del experimento

In [6]:
# Tabla unificada con todas las corridas del experimento (24 runs totales).
# La renombramos a param1/param2/param3 para tener una estructura consistente
# entre las 3 técnicas → más fácil de poner en el PPT.
best_rf  = df_rf.loc[df_rf['F1 (weighted)'].idxmax()]
best_svm = df_svm.loc[df_svm['F1 (weighted)'].idxmax()]
best_mlp = df_mlp.loc[df_mlp['F1 (weighted)'].idxmax()]

print('=== Mejor configuración por técnica (por F1 weighted) ===')
print(f"\nRF  → Acc={best_rf['Accuracy']} | F1={best_rf['F1 (weighted)']}")
print(f"      n_estimators={best_rf['n_estimators']}, max_depth={best_rf['max_depth']}, min_samples_split={best_rf['min_samples_split']}")
print(f"\nSVM → Acc={best_svm['Accuracy']} | F1={best_svm['F1 (weighted)']}")
print(f"      C={best_svm['C']}, kernel={best_svm['kernel']}, gamma={best_svm['gamma']}")
print(f"\nMLP → Acc={best_mlp['Accuracy']} | F1={best_mlp['F1 (weighted)']}")
print(f"      hidden={best_mlp['hidden_layers']}, activation={best_mlp['activation']}, lr={best_mlp['learning_rate']}")

# Tabla consolidada para el PPT
df_exp = pd.concat([
    df_rf[['Técnica','n_estimators','max_depth','min_samples_split','Accuracy','F1 (weighted)']].rename(
        columns={'n_estimators':'param1','max_depth':'param2','min_samples_split':'param3'}),
    df_svm[['Técnica','C','kernel','gamma','Accuracy','F1 (weighted)']].rename(
        columns={'C':'param1','kernel':'param2','gamma':'param3'}),
    df_mlp[['Técnica','hidden_layers','activation','learning_rate','Accuracy','F1 (weighted)']].rename(
        columns={'hidden_layers':'param1','activation':'param2','learning_rate':'param3'})
], ignore_index=True)

df_exp.to_csv(RESULTS_PATH + 'experiment_table.csv', index=False)
print(f'\n✅ Tabla guardada en results/experiment_table.csv')
df_exp

=== Mejor configuración por técnica (por F1 weighted) ===

RF  → Acc=0.6815 | F1=0.6864
      n_estimators=200, max_depth=10, min_samples_split=10

SVM → Acc=0.6631 | F1=0.6694
      C=1, kernel=rbf, gamma=scale

MLP → Acc=0.7347 | F1=0.6867
      hidden=(100, 50), activation=tanh, lr=0.01

✅ Tabla guardada en results/experiment_table.csv


,Técnica,param1,param2,param3,Accuracy,F1 (weighted)
0,RF,100,10,5,0.6775,0.6833
1,RF,100,10,10,0.6817,0.6861
2,RF,100,20,5,0.6774,0.6823
3,RF,100,20,10,0.6778,0.6840
4,RF,200,10,5,0.6792,0.6844
5,RF,200,10,10,0.6815,0.6864
6,RF,200,20,5,0.6788,0.6837
7,RF,200,20,10,0.6779,0.6842
8,SVM,1,rbf,scale,0.6631,0.6694
9,SVM,1,rbf,auto,0.6632,0.6694


## 7. Visualización del experimento

In [7]:
plot_experiment_bars(df_rf, df_svm, df_mlp, RESULTS_PATH + 'experimento_accuracy.png')

## 8. Análisis comparativo
*(Completar con el análisis escrito basado en los resultados obtenidos)*

- **Mejor técnica en accuracy:** [completar]
- **Técnica más sensible a hiperparámetros:** [completar]
- **Hiperparámetro más influyente por técnica:** [completar]
- **Conclusión general:** [completar]